<a href="https://colab.research.google.com/github/jyizheng/my-study/blob/main/pytorch-code/NanoGPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [66]:
import math
import inspect
from dataclasses import dataclass

import torch
import torch.nn as nn
from torch.nn import functional as F

import tiktoken

In [67]:
class LayerNorm(nn.Module):
    """ LayerNorm but with an optional bias. PyTorch doesn't support simply bias=False """

    def __init__(self, ndim, bias):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(ndim))
        self.bias = nn.Parameter(torch.zeros(ndim)) if bias else None

    def forward(self, input):
        return F.layer_norm(input, self.weight.shape, self.weight, self.bias, 1e-5)

class CausalSelfAttention(nn.Module):

    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        # key, query, value projections for all heads, but in a batch
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        # output projection
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        # regularization
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.dropout = config.dropout
        # flash attention make GPU go brrrrr but support is only in PyTorch >= 2.0
        self.flash = hasattr(torch.nn.functional, 'scaled_dot_product_attention')
        if not self.flash:
            print("WARNING: using slow attention. Flash Attention requires PyTorch >= 2.0")
            # causal mask to ensure that attention is only applied to the left in the input sequence
            self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                                        .view(1, 1, config.block_size, config.block_size))

    def forward(self, x, key_cache=None, value_cache=None, layer_past=None):
        B, T, C = x.size() # batch size, sequence length, embedding dimensionality (n_embd)

        # calculate query, key, values for all heads in batch and move head forward to be the batch dim
        q, k, v  = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) # (B, nh, T, hs)

        if layer_past is not None:
            past_key, past_value = layer_past
            k = torch.cat((past_key, k), dim=-2)
            v = torch.cat((past_value, v), dim=-2)

        present = (k, v) #tuple to be returned to cache

        # causal self-attention; Self-attend: (B, nh, T, hs) x (B, nh, hs, T) -> (B, nh, T, T)
        if self.flash:
            # efficient attention using Flash Attention CUDA kernels
            y = torch.nn.functional.scaled_dot_product_attention(q, k, v, attn_mask=None, dropout_p=self.dropout if self.training else 0, is_causal=True)
        else:
            # manual implementation of attention
            att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
            att = att.masked_fill(self.bias[:,:,:T,:k.size(-2)] == 0, float('-inf')) # Adjusted mask
            att = F.softmax(att, dim=-1)
            att = self.attn_dropout(att)
            y = att @ v # (B, nh, T, T) x (B, nh, T, hs) -> (B, nh, T, hs)
        y = y.transpose(1, 2).contiguous().view(B, T, C) # re-assemble all head outputs side by side

        # output projection
        y = self.resid_dropout(self.c_proj(y))
        return y, present

class MLP(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.c_fc    = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.gelu    = nn.GELU()
        self.c_proj  = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        x = self.dropout(x)
        return x

class Block(nn.Module):
    """ Transformer block """

    def __init__(self, config):
        super().__init__()
        self.ln_1 = LayerNorm(config.n_embd, bias=config.bias)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = LayerNorm(config.n_embd, bias=config.bias)
        self.mlp = MLP(config)

    def forward(self, x, layer_past=None):
        attn_output, present = self.attn(self.ln_1(x), layer_past=layer_past)
        x = x + attn_output
        x = x + self.mlp(self.ln_2(x))
        return x, present


@dataclass
class GPTConfig:
    block_size: int = 1024
    vocab_size: int = 50304 # GPT-2 vocab_size of 50257, padded up to nearest multiple of 64 for efficiency
    n_layer: int = 12
    n_head: int = 12
    n_embd: int = 768
    dropout: float = 0.0
    bias: bool = True # True: bias in Linears and LayerNorms, like GPT-2. False: a bit better and faster

class GPT(nn.Module):

    def __init__(self, config: GPTConfig):
        super().__init__()
        assert config.vocab_size is not None
        assert config.block_size is not None
        self.config = config

        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd),
            wpe = nn.Embedding(config.block_size, config.n_embd),
            drop = nn.Dropout(config.dropout),
            h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f = LayerNorm(config.n_embd, bias=config.bias),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        # with weight tying when using torch.compile() some warnings get generated:
        # "UserWarning: functional_call was passed multiple values for tied weights.
        # This behavior is deprecated and will be an error in future versions"
        # not 100% sure what this is, so far seems to be harmless. TODO investigate
        self.transformer.wte.weight = self.lm_head.weight # https://paperswithcode.com/method/weight-tying

        # init all weights
        self.apply(self._init_weights)
        # apply special scaled init to the residual projections, per GPT-2 paper
        for pn, p in self.named_parameters():
            if pn.endswith('c_proj.weight'):
                torch.nn.init.normal_(p, mean=0.0, std=0.02/math.sqrt(2 * config.n_layer))

        # report number of parameters
        print("number of parameters: %.2fM" % (self.get_num_params()/1e6,))

    def get_num_params(self, non_embedding=True):
        """
        Return the number of parameters in the model.
        For non-embedding count (default), the position embeddings get subtracted.
        The token embeddings would too, except due to the parameter sharing these
        params are actually used as weights in the final layer, so we include them.
        """
        n_params = sum(p.numel() for p in self.parameters())
        if non_embedding:
            n_params -= self.transformer.wpe.weight.numel()
        return n_params

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None, past=None):
        device = idx.device
        b, t = idx.size()

        # if past is not None, it means we are in the generation loop and only processing the last token
        if past is not None:
            t = idx.size(1)
            # only the last token is passed in the generation loop
            # so we need to adjust the position accordingly
            past_length = past[0][0].size(-2)
            pos = torch.tensor([past_length], dtype=torch.long, device=device)
        else:
            assert t <= self.config.block_size, f"Cannot forward sequence of length {t}, block size is only {self.config.block_size}"
            pos = torch.arange(0, t, dtype=torch.long, device=device) # shape (t)


        # forward the GPT model itself
        tok_emb = self.transformer.wte(idx) # token embeddings of shape (b, t, n_embd)
        pos_emb = self.transformer.wpe(pos) # position embeddings of shape (t, n_embd)
        x = self.transformer.drop(tok_emb + pos_emb)

        presents = []
        for i, block in enumerate(self.transformer.h):
            past_block = past[i] if past is not None else None
            x, present = block(x, layer_past=past_block)
            presents.append(present)

        x = self.transformer.ln_f(x)

        if targets is not None:
            # if we are given some desired targets also calculate the loss
            logits = self.lm_head(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)
        else:
            # inference-time mini-optimization: only forward the lm_head on the very last position
            logits = self.lm_head(x[:, -1, :]) # note: using list [-1] to preserve the time dim
            loss = None # During inference, we don't calculate loss

        return logits, loss, presents

    @classmethod
    def from_pretrained(cls, model_type, override_args=None):
        assert model_type in {'gpt2', 'gpt2-medium', 'gpt2-large', 'gpt2-xl'}
        override_args = override_args or {} # default to empty dict
        # only dropout can be overridden see more notes below
        assert all(k == 'dropout' for k in override_args)
        from transformers import GPT2LMHeadModel
        print("loading weights from pretrained gpt: %s" % model_type)

        # n_layer, n_head and n_embd are determined from model_type
        config_args = {
            'gpt2':         dict(n_layer=12, n_head=12, n_embd=768),  # 124M params
            'gpt2-medium':  dict(n_layer=24, n_head=16, n_embd=1024), # 350M params
            'gpt2-large':   dict(n_layer=36, n_head=20, n_embd=1280), # 774M params
            'gpt2-xl':      dict(n_layer=48, n_head=25, n_embd=1600), # 1558M params
        }[model_type]
        print("forcing vocab_size=50257, block_size=1024, bias=True")
        config_args['vocab_size'] = 50257 # always 50257 for GPT model checkpoints
        config_args['block_size'] = 1024 # always 1024 for GPT model checkpoints
        config_args['bias'] = True # always True for GPT model checkpoints
        # we can override the dropout rate, if desired
        if 'dropout' in override_args:
            print(f"overriding dropout rate to {override_args['dropout']}")
            config_args['dropout'] = override_args['dropout']
        # create a from-scratch initialized minGPT model
        config = GPTConfig(**config_args)
        model = GPT(config)
        sd = model.state_dict()
        sd_keys = sd.keys()
        sd_keys = [k for k in sd_keys if not k.endswith('.attn.bias')] # discard this mask / buffer, not a param

        # init a huggingface/transformers model
        model_hf = GPT2LMHeadModel.from_pretrained(model_type)
        sd_hf = model_hf.state_dict()

        # copy while ensuring all of the parameters are aligned and match in names and shapes
        sd_keys_hf = sd_hf.keys()
        sd_keys_hf = [k for k in sd_keys_hf if not k.endswith('.attn.masked_bias')] # ignore these, just a buffer
        sd_keys_hf = [k for k in sd_keys_hf if not k.endswith('.attn.bias')] # same, just the mask (buffer)
        transposed = ['attn.c_attn.weight', 'attn.c_proj.weight', 'mlp.c_fc.weight', 'mlp.c_proj.weight']
        # basically the openai checkpoints use a "Conv1D" module, but we only want to use a vanilla Linear
        # this means that we have to transpose these weights when we import them
        assert len(sd_keys_hf) == len(sd_keys), f"mismatched keys: {len(sd_keys_hf)} != {len(sd_keys)}"
        for k in sd_keys_hf:
            if any(k.endswith(w) for w in transposed):
                # special treatment for the Conv1D weights we need to transpose
                assert sd_hf[k].shape[::-1] == sd[k].shape
                with torch.no_grad():
                    sd[k].copy_(sd_hf[k].t())
            else:
                # vanilla copy over the other parameters
                assert sd_hf[k].shape == sd[k].shape
                with torch.no_grad():
                    sd[k].copy_(sd_hf[k])

        return model

# Problem 1: Write a decoding loop

I want to sample from my (completely untrained) GPT.

Write me a function, `sample_prompts(model, tokenizer, prompt, max_tokens)` to do this.

input: some string
output: certain amount of tokens --> those tokens appended to input prompt

In [76]:
small_gpt = GPT.from_pretrained("gpt2")
enc = tiktoken.get_encoding("gpt2")
tokenizer = tiktoken.get_encoding("gpt2")

def sample_prompts(model, tokenizer, prompt, max_tokens, top_k=None, top_p=None, temperature=1.0):
    """
    Samples tokens from a model given a prompt using greedy, Top-K, or Top-P decoding with KV caching.

    Args:
      model: The model to sample from.
      tokenizer: The tokenizer to use.
      prompt: The input prompt string.
      max_tokens: The maximum number of tokens to generate.
      top_k: If not None, consider only the top_k most likely tokens.
      top_p: If not None, consider tokens with cumulative probability up to top_p.
      temperature: Controls the randomness of the sampling. Higher values mean more randomness.

    Returns:
      The generated text including the original prompt.
    """
    model.eval()  # Set the model to evaluation mode
    encoded_prompt = tokenizer.encode(prompt)
    input_ids = torch.tensor(encoded_prompt, dtype=torch.long).unsqueeze(0) # Add batch dimension

    generated_tokens = input_ids
    past_key_values = None

    for i in range(max_tokens):
        print(f"--- Iteration {i+1} ---")
        print(f"Input shape to model: {generated_tokens.shape}")
        if past_key_values is not None:
            print(f"past_key_values shape (first layer k): {past_key_values[0][0].shape}")


        # Get the model's predictions for the next token
        with torch.no_grad():
            if past_key_values is None:
                logits, loss, past_key_values = model(generated_tokens)
                print(f"Initial model call output shapes: logits={logits.shape}, loss={loss}, past_key_values structure: {[ (k.shape, v.shape) for k,v in past_key_values ]}")
                # For the initial call, logits is (B, T, vocab_size), we take the last token's logits
                # last_token_logits = logits[:, -1, :] / temperature # Apply temperature - This was the incorrect line
                last_token_logits = logits / temperature # Correct indexing for (B, vocab_size)


            else:
                # In the generation loop, we only feed the last generated token
                # and pass the past_key_values to the model
                logits, loss, past_key_values = model(generated_tokens[:, -1].unsqueeze(-1), past=past_key_values)
                print(f"Subsequent model call output shapes: logits={logits.shape}, loss={loss}, past_key_values structure: {[ (k.shape, v.shape) for k,v in past_key_values ]}")
                # For subsequent calls with KV cache, logits is (B, 1, vocab_size)
                #last_token_logits = logits[:, 0, :] / temperature # Apply temperature
                # 修复：
                # The model's forward pass optimization ALWAYS returns [B, vocab_size] logits.
                last_token_logits = logits / temperature # Apply temperature


        print(f"Last token logits shape: {last_token_logits.shape}")


        # Apply Top-K sampling
        if top_k is not None:
            v, _ = torch.topk(last_token_logits, min(top_k, last_token_logits.size(-1)))
            last_token_logits[last_token_logits < v[:, [-1]]] = float('-inf')
            print(f"After Top-K, last token logits shape: {last_token_logits.shape}")

        # Apply Top-P sampling
        if top_p is not None:
            sorted_logits, sorted_indices = torch.sort(last_token_logits, descending=True)
            cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)

            # Remove tokens with cumulative probability above the threshold
            sorted_indices_to_remove = cumulative_probs > top_p
            # Shift the indices to the right to keep the first token above the threshold
            sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
            sorted_indices_to_remove[..., 0] = 0

            indices_to_remove = sorted_indices[sorted_indices_to_remove]
            last_token_logits[:, indices_to_remove] = float('-inf')
            print(f"After Top-P, last token logits shape: {last_token_logits.shape}")


        # Sample from the filtered distribution
        probs = F.softmax(last_token_logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        print(f"Next token shape: {next_token.shape}")


        # Append the predicted token to the generated sequence
        generated_tokens = torch.cat((generated_tokens, next_token), dim=1)
        print(f"Generated tokens shape: {generated_tokens.shape}")


    # Decode the generated tokens back to text
    generated_text = tokenizer.decode(generated_tokens[0].tolist())

    return generated_text


prompt = "Hello, my name is"
generated_text = sample_prompts(small_gpt, tokenizer, prompt, max_tokens=20)
print(generated_text)

loading weights from pretrained gpt: gpt2
forcing vocab_size=50257, block_size=1024, bias=True
number of parameters: 123.65M
--- Iteration 1 ---
Input shape to model: torch.Size([1, 5])
Initial model call output shapes: logits=torch.Size([1, 50257]), loss=None, past_key_values structure: [(torch.Size([1, 12, 5, 64]), torch.Size([1, 12, 5, 64])), (torch.Size([1, 12, 5, 64]), torch.Size([1, 12, 5, 64])), (torch.Size([1, 12, 5, 64]), torch.Size([1, 12, 5, 64])), (torch.Size([1, 12, 5, 64]), torch.Size([1, 12, 5, 64])), (torch.Size([1, 12, 5, 64]), torch.Size([1, 12, 5, 64])), (torch.Size([1, 12, 5, 64]), torch.Size([1, 12, 5, 64])), (torch.Size([1, 12, 5, 64]), torch.Size([1, 12, 5, 64])), (torch.Size([1, 12, 5, 64]), torch.Size([1, 12, 5, 64])), (torch.Size([1, 12, 5, 64]), torch.Size([1, 12, 5, 64])), (torch.Size([1, 12, 5, 64]), torch.Size([1, 12, 5, 64])), (torch.Size([1, 12, 5, 64]), torch.Size([1, 12, 5, 64])), (torch.Size([1, 12, 5, 64]), torch.Size([1, 12, 5, 64]))]
Last token log

# Problem 2: Optimize it!

The loop is verrrrry slow. Why? What can we do to fix it?

# Problem 3: Add top K and top P sampling

Top K sampling: Only consider the top K logprobs.

Top P sampling: Only consider the logprobs that sum to P probability.

Greedy decoding is implemented in the `sample_prompts` function when `top_k` and `top_p` are not specified.